# NumPy MLP ile spektral maskeleme

Ömer Altın ve Muhammed Aslan — EE491–EE492

Önceki MLP, temiz log-genlik spektrumunu doğrudan tahmin ediyordu. Hata azalmasına
rağmen örnek sesin SNR'si yaklaşık 5 dB'den 3.95 dB'ye düştü.

Bu sürümde aynı **257 → 64 → 32 → 257** MLP bir frekans maskesi tahmin ediyor.
SNN, PyTorch veya hazır bir sinir ağı eğitim kütüphanesi kullanmıyoruz.
Girdi verileri ve train/validation/test ayrımı önceki paketle aynı.

Bu defteri ayrı bir Colab not defteri olarak açın. Veri klasörü bulunmazsa ilk hücre
sizden önceki **DNS_MLP_Baslangic.zip** dosyasını ister. Bu güncelleme paketinde ses
veri setinin tamamı tekrar bulunmaz.

In [ ]:
from pathlib import Path
import csv, json, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import ShortTimeFFT

ROOT = Path('/content/DNS_MLP_Baslangic')
if (Path.cwd() / 'pairs.csv').exists():
    ROOT = Path.cwd()

if not (ROOT / 'pairs.csv').exists():
    from google.colab import files
    import zipfile, io
    print('Dosyaları Seç düğmesine basıp önceki DNS_MLP_Baslangic.zip dosyasını seçin.')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
        assert 'DNS_MLP_Baslangic/pairs.csv' in z.namelist(), 'Önceki VERİ ZIP dosyasını seçin.'
        z.extractall('/content')
    ROOT = Path('/content/DNS_MLP_Baslangic')

assert (ROOT / 'pairs.csv').exists()
OUTPUT_DIR = ROOT / 'mlp_mask_v2'
OUTPUT_DIR.mkdir(exist_ok=True)

with (ROOT / 'pairs.csv').open(encoding='utf-8', newline='') as f:
    pairs = list(csv.DictReader(f))

transform = ShortTimeFFT(np.hanning(512), hop=128, fs=16000, fft_mode='onesided')
print('Veri hazır:', ROOT)
for split in ['train', 'validation', 'test']:
    print(split, sum(r['split'] == split for r in pairs), 'çift')

## Yeni hedef

Modelin çıktısı her zaman-frekans noktası için 0 ile 1 arasında bir katsayıdır:

$$\widehat S(t,f)=M(t,f)\,Y(t,f).$$

Burada Y gürültülü kompleks STFT, M tahmin edilen maske, tahmin edilen S ise çıkıştır.
M=1 o bileşeni korur; M=0 o bileşeni bastırır. Bu katsayı bir sınıf olasılığı değildir.
Model frekans genliğini artırmaz; yine de konuşmayı bozma ihtimali tamamen ortadan kalkmaz.

Girdide log-genlik ve yalnız eğitimden öğrenilen normalizasyon kullanılıyor.
Hedef ise fazı dikkate alan spektral yaklaşımla hazırlanıyor:

$$T=\frac{\mathrm{Re}(S\overline{Y})}{|Y|^2},\qquad P=|Y|^2.$$

S eğitimdeki temiz referanstır. Sayısal kararlılık için paydada alt sınır kullanıyoruz.
Referans T, 0–1 aralığı dışına çıkabilir; ağın uyguladığı M ise sigmoid ile 0–1 arasındadır.
Bu kısıtlı bir yaklaşımdır.

Kayıp, P ile ağırlıklandırılmış maske karesel hatasıdır. Sıfır olmayan frekanslarda,
kompleks spektral hatadan maskeden bağımsız bir sabit ve ölçek kadar farklıdır.
Önceki **log-spektrum MSE** ile bu kaybın sayısal değerlerini karşılaştırmayın.
İki modeli aynı kayıtlardaki dalga biçimi SNR'siyle karşılaştıracağız.

In [ ]:
def read_audio(relative):
    fs,x=wavfile.read(ROOT/relative)
    assert fs==16000 and x.shape==(80000,)
    return x.astype(np.float64)/32768


def prepare(split):
    rows=[r for r in pairs if r['split']==split]
    frames=transform.stft(read_audio(rows[0]['noisy_path'])).shape[1]
    X=np.empty((len(rows)*frames,257),dtype=np.float32)
    T=np.empty_like(X); P=np.empty_like(X)
    for i,r in enumerate(rows):
        s=transform.stft(read_audio(r['clean_path']))
        y=transform.stft(read_audio(r['noisy_path']))
        power=np.abs(y)**2
        target=np.real(s*np.conj(y))/np.maximum(power,1e-8)
        ix=slice(i*frames,(i+1)*frames)
        X[ix]=np.log1p(np.abs(y)).T
        T[ix]=target.T
        P[ix]=power.T
    print('Prepared',split,X.shape,flush=True)
    return X,T,P

## İleri ve geri yayılım

Gizli katmanlar ReLU, son katman sigmoid kullanır. W1 ve W2 rastgele başlar.
W3 başlangıçta sıfırdır; b3 sayesinde başlangıç maskesi 0.9 olur. Böylece ilk çıktı
girişin büyük kısmını korur. Gizli katmanların farklı rastgele özellikleri simetriyi kırar.

Geri yayılımda önce ağırlıklı kaybın maskeye göre türevini, ardından sigmoid türevi
M(1−M)'yi kullanıyoruz. Sonraki ReLU ve matris türevleri önceki ağdaki mantığı izler.

In [ ]:
def initialize():
    rng=np.random.default_rng(42)
    return {
      'W1':(rng.standard_normal((64,257))*np.sqrt(2/257)).astype(np.float32),
      'b1':np.zeros((64,1),dtype=np.float32),
      'W2':(rng.standard_normal((32,64))*np.sqrt(2/64)).astype(np.float32),
      'b2':np.zeros((32,1),dtype=np.float32),
      'W3':np.zeros((257,32),dtype=np.float32),
      'b3':np.full((257,1),np.log(0.9/0.1),dtype=np.float32)}


def forward_mask(x,p):
    z1=x@p['W1'].T+p['b1'].T;a1=np.maximum(z1,0)
    z2=a1@p['W2'].T+p['b2'].T;a2=np.maximum(z2,0)
    z3=a2@p['W3'].T+p['b3'].T
    mask=np.exp(-np.logaddexp(0,-z3))
    return mask,(x,z1,a1,z2,a2)


def objective(mask,target,power):
    return float(np.sum(power*(mask-target)**2,dtype=np.float64)/(np.sum(power,dtype=np.float64)+1e-12))


def backward_mask(mask,target,power,cache,p):
    x,z1,a1,z2,a2=cache
    dmask=2*power*(mask-target)/(np.sum(power,dtype=np.float64)+1e-12)
    dz3=dmask*mask*(1-mask)
    dz2=(dz3@p['W3'])*(z2>0)
    dz1=(dz2@p['W2'])*(z1>0)
    return {'W3':dz3.T@a2,'b3':dz3.sum(axis=0,keepdims=True).T,
      'W2':dz2.T@a1,'b2':dz2.sum(axis=0,keepdims=True).T,
      'W1':dz1.T@x,'b1':dz1.sum(axis=0,keepdims=True).T}


def evaluate(X,T,P,p):
    numerator=denominator=0.
    for start in range(0,len(X),1024):
        sl=slice(start,start+1024)
        pred,_=forward_mask(X[sl],p)
        numerator+=float(np.sum(P[sl]*(pred-T[sl])**2,dtype=np.float64))
        denominator+=float(np.sum(P[sl],dtype=np.float64))
    return numerator/(denominator+1e-12)

## Eğitim ve model seçimi

En fazla 30 epoch, batch 512 ve learning rate 0.1 kullanılır. Her çalıştırma yeni bir
ağ başlatır. En düşük doğrulama kaybını veren ağırlıklar saklanır; son epoch'un
ağırlıklarının en iyi olduğu varsayılmaz. Test verisi ağırlık veya epoch seçmek için kullanılmaz.

Eğitim çıktıları mlp_mask_v2 klasörüne kaydedilir. Çalışma süresi işlemciye bağlıdır;
her epoch'un süresi ekranda yazılır.

In [ ]:
def run_training():
    X,T,P=prepare('train');V,U,Q=prepare('validation')
    mean=X.mean(axis=0,keepdims=True,dtype=np.float64).astype(np.float32)
    std=np.maximum(X.std(axis=0,keepdims=True,dtype=np.float64).astype(np.float32),1e-4)
    X-=mean;X/=std;V-=mean;V/=std
    p=initialize();best={k:v.copy() for k,v in p.items()}
    best_loss=evaluate(V,U,Q,p);best_epoch=0
    rng=np.random.default_rng(123);history=[]
    # Settings fixed before inspecting test outcomes.
    for epoch in range(1,31):
        tic=time.perf_counter();order=rng.permutation(len(X))
        for start in range(0,len(X),512):
            ii=order[start:start+512]
            prediction,cache=forward_mask(X[ii],p)
            grads=backward_mask(prediction,T[ii],P[ii],cache,p)
            for key in p:p[key]-=0.1*grads[key]
        train=evaluate(X,T,P,p);val=evaluate(V,U,Q,p)
        history.append({'epoch':epoch,'train':train,'validation':val})
        if val<best_loss:
            best_loss=val;best_epoch=epoch;best={k:v.copy() for k,v in p.items()}
        print('EPOCH',epoch,'train',round(train,6),'val',round(val,6),'seconds',round(time.perf_counter()-tic,2),flush=True)
    np.savez(OUTPUT_DIR/'mlp_mask_weights.npz',**best,x_mean=mean,x_std=std,
        best_epoch=np.array(best_epoch),best_val_loss=np.array(best_loss))
    (OUTPUT_DIR/'mask_training_history.json').write_text(json.dumps(history,indent=2))
    print('BEST',best_epoch,best_loss,flush=True)
    return best,mean,std


mask_params, mask_mean, mask_std = run_training()

with np.load(OUTPUT_DIR / 'mlp_mask_weights.npz') as checkpoint:
    print('Seçilen epoch:', int(checkpoint['best_epoch']))

history = json.loads((OUTPUT_DIR / 'mask_training_history.json').read_text())
plt.figure(figsize=(8, 4))
plt.plot([h['epoch'] for h in history], [h['train'] for h in history], label='Training')
plt.plot([h['epoch'] for h in history], [h['validation'] for h in history], label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Power-weighted mask MSE')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mask_training_curve.png', dpi=150)
plt.show()

## Sese dönüştürme ve ölçüm

Tahmin edilen maskeyi gürültülü kompleks STFT ile çarpıp ters STFT uygularız.
Temiz ses yalnız değerlendirme için kullanılır; çıkarım fonksiyonuna verilmez.

SNR, temiz referans ile çıkış arasındaki hatayı ölçer; kalan gürültüyü ve konuşma
bozulmasını birlikte içerir. Ölçüm dinleme kazancı uygulanmadan önce yapılır.

In [ ]:
def enhance(noisy,p,mean,std):
    spec=transform.stft(noisy)
    features=np.log1p(np.abs(spec)).T.astype(np.float32)
    mask,_=forward_mask((features-mean)/std,p)
    output=transform.istft(mask.T*spec,k1=len(noisy))
    assert np.isfinite(output).all() and mask.min()>=0 and mask.max()<=1
    return output


def snr(reference,estimate):
    return float(10*np.log10((np.sum(reference**2)+1e-12)/(np.sum((reference-estimate)**2)+1e-12)))


def score(split,p,mean,std):
    results=[]
    for r in pairs:
        if r['split']!=split:continue
        clean=read_audio(r['clean_path']);noisy=read_audio(r['noisy_path'])
        enhanced=enhance(noisy,p,mean,std)
        before=snr(clean,noisy);after=snr(clean,enhanced)
        results.append({'clean_id':r['clean_id'],'snr':int(r['target_snr_db']),
          'before':before,'after':after,'improvement':after-before})
    for level in [0,5,10]:
        rows=[r for r in results if r['snr']==level]
        print(split,level,{k:float(np.mean([r[k] for r in rows])) for k in ['before','after','improvement']},flush=True)
    return results


validation_results = score('validation', mask_params, mask_mean, mask_std)
test_results = score('test', mask_params, mask_mean, mask_std)

for split, rows in [('validation', validation_results), ('test', test_results)]:
    with (OUTPUT_DIR / f'{split}_snr.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)

## Aynı örneği dinleme

Önceki karşılaştırmadaki ilk 5 dB test örneğini seçiyoruz; en iyi duyulan örneği
sonradan aramıyoruz. Üç kayda aynı oynatma kazancı uygulanır.

In [ ]:
from IPython.display import Audio, display

row = next(r for r in pairs if r['split'] == 'test' and r['target_snr_db'] == '5')
reference = read_audio(row['clean_path'])
noisy = read_audio(row['noisy_path'])
masked = enhance(noisy, mask_params, mask_mean, mask_std)

print('Sabit örnek:', row['clean_id'])
print('Giriş SNR:', snr(reference, noisy))
print('Maske çıkışı SNR:', snr(reference, masked))

signals = [('clean_reference', reference), ('noisy_input', noisy), ('mlp_mask_output', masked)]
peak = max(np.max(np.abs(x)) for _, x in signals)
gain = 0.95 / max(1.0, peak)
for label, signal in signals:
    print(label)
    playback = signal * gain
    display(Audio(playback, rate=16000, normalize=False))
    wavfile.write(OUTPUT_DIR / f'{label}.wav', 16000,
                 np.rint(playback * 32767).astype(np.int16))

## Sonuçları indirme

İlk sürümün test sonuçları daha önce görüldü. Buradaki karşılaştırma aynı küçük alt
kümede bir geliştirme deneyi olarak sunulmalı; yeni, kör bir benchmark sonucu olarak değil.
Sonraki değişiklikler doğrulama grubunda seçilmeli. Daha güçlü nihai genelleme iddiası
için daha önce kullanılmamış konuşma ve gürültü kaynaklarıyla ek değerlendirme yapılabilir.

Bu küçük MLP'nin zaman belleği yok ve yalnızca 20 gürültü kaynağından yararlanıyor.
Yeni maske yöntemi her koşulda iyileşmeyi garanti etmez. Raporda SNR seviyelerini ayrı gösterin.

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive('/content/MLP_Mask_Sonuclarim', 'zip', OUTPUT_DIR)
files.download(archive)

## Kaynaklar

- Microsoft DNS Challenge 3: https://github.com/microsoft/DNS-Challenge/blob/master/README-DNS3.md
- H. Erdoğan et al., Phase-sensitive and recognition-boosted speech separation using deep recurrent neural networks, ICASSP 2015, doi:10.1109/ICASSP.2015.7178061.
  https://research.sabanciuniv.edu/id/eprint/28858/
- Bu çalışma makaledeki RNN'nin yeniden üretimi değildir; maskeleme ve sinyal yaklaşımı
  fikrini küçük bir NumPy MLP ile kullanan eğitim amaçlı bir deneydir.